In [1]:
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2619, done.
remote: Total 2619 (delta 0), reused 0 (delta 0), pack-reused 2619 (from 1)
Receiving objects: 100% (2619/2619), 8.24 MiB | 26.27 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.


In [2]:
%cd pytorch-CycleGAN-and-pix2pix

/kaggle/working/pytorch-CycleGAN-and-pix2pix


In [3]:
!ls

CycleGAN.ipynb	docs		 LICENSE  pix2pix.ipynb  test.py
data		environment.yml  models   README.md	 train.py
datasets	imgs		 options  scripts	 util


In [4]:
import os
print(os.listdir())  # Menampilkan semua folder dan file di direktori saat ini

['scripts', 'docs', 'util', 'train.py', '.replit', 'LICENSE', 'imgs', 'data', 'models', '.git', 'datasets', 'README.md', 'pix2pix.ipynb', 'environment.yml', 'CycleGAN.ipynb', 'options', 'test.py', '.gitignore']


In [5]:
!ls -F

CycleGAN.ipynb	docs/		 LICENSE   pix2pix.ipynb  test.py
data/		environment.yml  models/   README.md	  train.py
datasets/	imgs/		 options/  scripts/	  util/


In [6]:
!pip install dominate visdom

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408195 sha256=f0ab05ff506de4aaab0e63e72919fc93d81e670af6ad3c258a7743129acc74ff
  Stored in directory: /root/.cache/pip/wheels/37/6c/38/64eeaa310e325aacda723e6df1f79ab5e9f31ba195264e04a8
Successfully built visdom


In [7]:
# DOMINATE?
# A Python library for automatically generating HTML pages,
# typically used in projects like CycleGAN to display training results.

In [8]:
import wandb
!wandb login 390063a1a72a99da508afc81c6b099a74c5c6afc

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Roboflow Dataset

In [9]:
!pip install roboflow

import roboflow
# Inisialisasi API Roboflow
dataset_name = "traintestrgbndvi-ilx7x"
dataset_ver = 1
rf = roboflow.Roboflow(api_key="uws7Wv4LJKXm3PrYyg9a")
project = rf.workspace("datasetta-ubg4x").project(dataset_name)
version = project.version(dataset_ver)

# Download dataset dalam format yang sesuai
dataset = version.download("folder", location="/kaggle/working/cyclegan-dataset")

print("Dataset downloaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 99.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, 


Extracting Dataset Version Zip to /kaggle/working/cyclegan-dataset in folder:: 100%|██████████| 608/608 [00:00<00:00, 3725.39it/s]

Dataset downloaded successfully!


### Split Data

In [10]:
import os
from sklearn.model_selection import train_test_split
from shutil import copyfile

# Define dataset name and version
dataset_name = 'cyclegan-dataset'
dataset_ver = '1.0'

# Set the root folder
root_folder = '/kaggle/working/' + dataset_name
input_folder = os.path.join(root_folder, 'train')
# Correct the output_folder path to reflect the actual location of the cloned repository
output_folder = '/kaggle/working/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x'

# Verify input folder
if not os.path.exists(input_folder):
    raise FileNotFoundError(f"Input folder '{input_folder}' not found.")

# Create output folders if they don't exist
for folder_name in ['trainA', 'trainB', 'testA', 'testB']:
    folder_path = os.path.join(output_folder, folder_name)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# Define paths for ndvi and rgb
ndvi_path = os.path.join(input_folder, 'ndvi')
rgb_path = os.path.join(input_folder, 'rgb')

# Verify subfolders
if not os.path.exists(ndvi_path):
    raise FileNotFoundError(f"'ndvi' folder not found at {ndvi_path}")
if not os.path.exists(rgb_path):
    raise FileNotFoundError(f"'rgb' folder not found at {rgb_path}")

# List all files in the input folder for each category
ndvi_files = [f for f in os.listdir(ndvi_path) if f.endswith('.jpg')]
rgb_files = [f for f in os.listdir(rgb_path) if f.endswith('.jpg')]

# Split the data into training and test sets for both ndvi and rgb
rgb_train, rgb_test, ndvi_train, ndvi_test = train_test_split(rgb_files, ndvi_files, test_size=0.2, random_state=2024)

# Copy files to the corresponding folders
for file_name in ndvi_train:
    source_path = os.path.join(ndvi_path, file_name)
    destination_path = os.path.join(output_folder, 'trainB', file_name)
    copyfile(source_path, destination_path)

for file_name in rgb_train:
    source_path = os.path.join(rgb_path, file_name)
    destination_path = os.path.join(output_folder, 'trainA', file_name)
    copyfile(source_path, destination_path)

for file_name in ndvi_test:
    source_path = os.path.join(ndvi_path, file_name)
    destination_path = os.path.join(output_folder, 'testB', file_name)
    copyfile(source_path, destination_path)

for file_name in rgb_test:
    source_path = os.path.join(rgb_path, file_name)
    destination_path = os.path.join(output_folder, 'testA', file_name)
    copyfile(source_path, destination_path)

print("Data split and saved successfully.")

Data split and saved successfully.


# CycleGAN Train

### Train (ORIGINAL CODE)

In [11]:
# exp_name = 'traincyclegan'
# !python /content/pytorch-CycleGAN-and-pix2pix/train.py 
# --use_wandb 
# --wandb_project_name='cyclegan' 
# --gan_mode vanilla 
# --load_size 416 
# --crop_size 416 
# --preprocess 'none' 
# --dataroot /content/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x 
# --name $exp_name 
# --model cycle_gan
     

### Train (MODIFIED & ADJUSTMENT CODE)

In [12]:
# exp_name = 'traincyclegan'
# !python /kaggle/working/pytorch-CycleGAN-and-pix2pix/train.py \
#     --use_wandb \
#     --wandb_project_name cyclegan \
#     --gan_mode vanilla \
#     --load_size 256 \
#     --crop_size 256 \
#     --preprocess none \
#     --dataroot /kaggle/working/pytorch-CycleGAN-and-pix2pix/datasets/datasetta-ubg4x \
#     --name $exp_name \
#     --model cycle_gan \
#     --n_epochs 100 \
#     --n_epochs_decay 100 \
#     --save_epoch_freq 5 \
#     --save_latest_freq 1000 \
#     --display_freq 1000 \
#     --print_freq 100 \
#     --batch_size 1 \
#     --num_threads 4

In [13]:
# !ls /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints/traincyclegan

In [14]:
# !zip -r checkpoint_backup_epoch41.zip \
# /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints/traincyclegan

### Check REPO (Major Collisions)

In [15]:
# # Go back to the working directory first
# %cd /kaggle/working

# # Now move everything from sub-folder to master
# !mv pytorch-CycleGAN-and-pix2pix/pytorch-CycleGAN-and-pix2pix/* pytorch-CycleGAN-and-pix2pix/
# !mv pytorch-CycleGAN-and-pix2pix/pytorch-CycleGAN-and-pix2pix/.git* pytorch-CycleGAN-and-pix2pix/ 2>/dev/null || true
# !rm -rf pytorch-CycleGAN-and-pix2pix/pytorch-CycleGAN-and-pix2pix

# # Verify it worked
# %cd pytorch-CycleGAN-and-pix2pix
# !ls -la

In [16]:
# !ls -la /kaggle/working/pytorch-CycleGAN-and-pix2pix/

In [17]:
# !ls -la /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints/

In [18]:
# !ls -la /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints/traincyclegan/

In [19]:
# !mkdir -p /kaggle/working/output
# !cp -r /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints/traincyclegan /kaggle/working/output/

In [20]:
# !ls -lh /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints/traincyclegan/

# !tar -czf /kaggle/working/traincyclegan_checkpoints.tar.gz \
#     -C /kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints traincyclegan

# !ls -lh /kaggle/working/traincyclegan_checkpoints.tar.gz

### Check .zip Checkpoint

In [21]:
# import os

# # Search for all .pth files (PyTorch model files)
# print("🔍 Searching for all checkpoint files (.pth)...\n")

# for root, dirs, files in os.walk('/kaggle/working'):
#     for file in files:
#         if file.endswith('.pth'):
#             full_path = os.path.join(root, file)
#             size_mb = os.path.getsize(full_path) / (1024 * 1024)
#             print(f"✅ Found: {full_path}")
#             print(f"   Size: {size_mb:.2f} MB\n")

# # Also check the checkpoints directory
# print("\n📂 Contents of checkpoints directory:")
# print("=" * 60)
# checkpoints_dir = '/kaggle/working/pytorch-CycleGAN-and-pix2pix/checkpoints'
# if os.path.exists(checkpoints_dir):
#     for item in os.listdir(checkpoints_dir):
#         item_path = os.path.join(checkpoints_dir, item)
#         if os.path.isdir(item_path):
#             print(f"\n📁 {item}/")
#             for subitem in os.listdir(item_path):
#                 subitem_path = os.path.join(item_path, subitem)
#                 if os.path.isfile(subitem_path):
#                     size_mb = os.path.getsize(subitem_path) / (1024 * 1024)
#                     print(f"   {subitem:<40} {size_mb:>8.2f} MB")
#         else:
#             size_mb = os.path.getsize(item_path) / (1024 * 1024)
#             print(f"   {item:<40} {size_mb:>8.2f} MB")
# else:
#     print("❌ Checkpoints directory not found!")